In this notebook, we do HPO of our DualEncoderFNO with Optuna using a 10% of our dataset. We use Multi-Objective optimization trying to minimize validation loss and number of model parameters.

### 1. Libraries

In [1]:
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

import torch
from torch.utils.data import DataLoader
from neuralop import LpLoss, H1Loss

from rve_analyzer import RVEDataset, DualEncoderFNO, Trainer

### 2. Configuration

In [2]:
from types import SimpleNamespace

cfg = SimpleNamespace(**{})

In [3]:
cfg.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {cfg.device}")

Device: cuda


In [4]:
cfg.h5_path = Path("../master_data/rve_run2.h5")
cfg.batch_size = 64
cfg.num_workers = 0
cfg.seed = 42
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

### 3. Datasets & DataLoader

In [5]:
cfg.in_memory = True
cfg.fraction = 0.10

train_dataset = RVEDataset(cfg.h5_path, split='train', in_memory=cfg.in_memory, fraction=cfg.fraction)
val_dataset   = RVEDataset(cfg.h5_path, split='val', in_memory=cfg.in_memory, fraction=cfg.fraction)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# Get dimensions
sample_xl, sample_xg, sample_y = train_dataset[0]
in_channels = sample_xl.shape[0]      # phase + nstatev + ...
out_channels = sample_y.shape[0]
n_macro = sample_xg.shape[0]

print(f"in_channels={in_channels}, out_channels={out_channels}, n_macro={n_macro}")


Loading 10% of 'train' split into RAM. This may take a moment...
Loading 10% of 'val' split into RAM. This may take a moment...
Train: 6000 | Val: 2000
in_channels=1, out_channels=3, n_macro=3


In [6]:
persistent_workers = True if cfg.num_workers > 0 else False
prefetch_factor = 4 if cfg.num_workers > 0 else None

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True, 
                          persistent_workers=persistent_workers, prefetch_factor=prefetch_factor)

val_loader   = DataLoader(val_dataset,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True, 
                          persistent_workers=persistent_workers, prefetch_factor=prefetch_factor)

### 4. HPO

In [7]:
cfg.hpo_dir = Path("../checkpoints/rve2_hpo")
cfg.hpo_dir.mkdir(parents=True, exist_ok=True)

In [8]:
# ================== CONFIG HPO ==================
cfg.hpo_epochs      = 40
cfg.n_trials        = 60
cfg.hpo_lr          = 1e-3
cfg.hpo_weight_decay = 1e-4

l2loss = LpLoss(d=2, p=2, reduction='mean')
h1loss = H1Loss(d=2, reduction='mean')

def objective(trial):
    """Multi-objective optimization: minimize validation loss AND number of parameters."""
    
    # HPO range
    n_modes          = trial.suggest_categorical("n_modes",          [8, 16, 24, 32])
    hidden_channels  = trial.suggest_categorical("hidden_channels",  [16, 32, 64])
    n_layers         = trial.suggest_categorical("n_layers",         [2, 4, 6])
    film_mlp_layers  = trial.suggest_categorical("film_mlp_layers",  [2, 4, 6])
    film_mlp_neurons = trial.suggest_categorical("film_mlp_neurons", [32, 64, 128])
    
    # Instance model
    model = DualEncoderFNO(
        in_channels         = in_channels,
        out_channels        = out_channels,
        n_macro             = n_macro,
        n_modes             = n_modes,
        hidden_channels     = hidden_channels,
        n_layers            =  n_layers,
        use_positional_grid = True,
        film_per_layer      = True,
        film_mlp_layers     = film_mlp_layers,
        film_mlp_neurons    = film_mlp_neurons,
    ).to(cfg.device)
    
    n_params = model.count_parameters()

    # Instance trainer
    trainer = Trainer(
        model       = model,
        loss_fun    = h1loss,
        val_metrics = {'l2': l2loss},
        wandb_log   = False,
        device      = cfg.device,
        save        = False,
        verbose     = False,
        min_delta   = 1e-7,
        max_grad_norm = 1.0
    )
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=cfg.hpo_lr, 
        weight_decay=cfg.hpo_weight_decay
    )
    
    # Training
    _ = trainer.fit(
        train_loader = train_loader,
        val_loader   = val_loader,
        epochs       = cfg.hpo_epochs,
        optimizer    = optimizer,
        patience     = cfg.hpo_epochs + 1,
        verbose      = False
    )
    
    best_val_loss = trainer.best_loss
    
    # Store extra info for later analysis
    trial.set_user_attr("best_val_loss", float(best_val_loss))
    trial.set_user_attr("n_params", int(n_params))
    trial.set_user_attr("n_params_M", round(n_params / 1e6, 2))
    
    print(f"Trial {trial.number:2d} | val_loss={best_val_loss:.6f} | params={n_params/1e6:.2f}M")
    
    # Return two objectives to minimize
    return best_val_loss, n_params


In [9]:
# ================== MULTI-OBJECTIVE STUDY WITH NSGA-II ==================
import optuna

study = optuna.create_study(
    directions=["minimize", "minimize"],                        # minimize both val_loss and params
    sampler=optuna.samplers.NSGAIISampler(population_size=30),  # efficient for multi-objective
    study_name="DualEncoderFNO_MultiObj"
)

print("Starting Multi-Objective HPO with NSGA-II...")
study.optimize(objective, n_trials=cfg.n_trials)

c:\GitHub\CEIA\feinn_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-20 19:51:41,447] A new study created in memory with name: DualEncoderFNO_MultiObj


Starting Multi-Objective HPO with NSGA-II...
DualEncoderFNO Training: 40 epochs



[I 2026-03-20 20:21:29,251] Trial 0 finished with values: [0.325247154712677, 17976515.0] and parameters: {'n_modes': 32, 'hidden_channels': 64, 'n_layers': 4, 'film_mlp_layers': 4, 'film_mlp_neurons': 128}.



 Training finished. Best epoch: 40 (Loss = 0.325247)
Trial  0 | val_loss=0.325247 | params=17.98M
DualEncoderFNO Training: 40 epochs



[I 2026-03-20 21:00:48,847] Trial 1 finished with values: [0.39521975231170653, 2084227.0] and parameters: {'n_modes': 8, 'hidden_channels': 64, 'n_layers': 6, 'film_mlp_layers': 2, 'film_mlp_neurons': 64}.


...


 Training finished. Best epoch: 40 (Loss = 0.489774)
Trial 57 | val_loss=0.489774 | params=0.98M
DualEncoderFNO Training: 40 epochs



[I 2026-03-21 13:18:40,516] Trial 58 finished with values: [0.5372320528030395, 969635.0] and parameters: {'n_modes': 24, 'hidden_channels': 16, 'n_layers': 6, 'film_mlp_layers': 2, 'film_mlp_neurons': 32}.



 Training finished. Best epoch: 40 (Loss = 0.537232)
Trial 58 | val_loss=0.537232 | params=0.97M
DualEncoderFNO Training: 40 epochs



[I 2026-03-21 13:59:30,399] Trial 59 finished with values: [0.3378478617668152, 15536003.0] and parameters: {'n_modes': 24, 'hidden_channels': 64, 'n_layers': 6, 'film_mlp_layers': 4, 'film_mlp_neurons': 128}.



 Training finished. Best epoch: 40 (Loss = 0.337848)
Trial 59 | val_loss=0.337848 | params=15.54M


In [10]:
# ================== PARETO FRONT ANALYSIS ==================
pareto_trials = study.best_trials

print(f"✅ Found {len(pareto_trials)} models on the Pareto Front\n")
print("Trial | Val Loss   | Params (M) | n_modes | hidden | layers | film_l | film_n")
print("-" * 85)

for t in pareto_trials:
    p = t.params
    print(f"{t.number:5d} | {t.values[0]:.6f} | {t.values[1]/1e6:7.2f}  | "
          f"{p['n_modes']:7} | {p['hidden_channels']:6} | "
          f"{p['n_layers']:6} | {p['film_mlp_layers']:6} | "
          f"{p['film_mlp_neurons']:6}")


✅ Found 15 models on the Pareto Front

Trial | Val Loss   | Params (M) | n_modes | hidden | layers | film_l | film_n
-------------------------------------------------------------------------------------
    1 | 0.395220 |    2.08  |       8 |     64 |      6 |      2 |     64
    2 | 0.712498 |    0.09  |       8 |     16 |      2 |      4 |    128
   15 | 0.549837 |    0.55  |       8 |     32 |      6 |      6 |     64
   17 | 0.379907 |    4.50  |      32 |     32 |      4 |      6 |     64
   20 | 0.461890 |    1.21  |      16 |     32 |      4 |      6 |     32
   21 | 0.569928 |    0.36  |       8 |     32 |      4 |      2 |     64
   26 | 0.312927 |   17.92  |      32 |     64 |      4 |      4 |     64
   27 | 0.335995 |    4.79  |      16 |     64 |      4 |      2 |     32
   28 | 0.659082 |    0.14  |       8 |     16 |      6 |      2 |     64
   29 | 0.572069 |    0.19  |       8 |     32 |      2 |      2 |    128
   32 | 0.480867 |    1.13  |      32 |     16 |      4 |

In [11]:
# ================== SAVE ALL TRIALS AND PARETO FRONT TO CSV ==================
import pandas as pd

# Save ALL trials (complete history)
df_all = study.trials_dataframe()

# Clean and rename columns for clarity
df_all = df_all.rename(columns={
    "values_0": "val_loss",
    "values_1": "n_params"
})
df_all["params_M"] = df_all["n_params"] / 1e6

# Save full history
all_trials_path = cfg.hpo_dir / "rve2_hpo_all_trials.csv"
df_all.to_csv(all_trials_path, index=False)

print(f"✅ All trials saved")
print(f"   → {len(df_all)} trials total")

# Save ONLY the Pareto Front
pareto_data = []
for t in study.best_trials:
    pareto_data.append({
        "trial": t.number,
        "val_loss": t.values[0],
        "params_M": t.values[1] / 1e6,
        **t.params
    })

pareto_df = pd.DataFrame(pareto_data)
pareto_df = pareto_df.sort_values("params_M")

pareto_path = cfg.hpo_dir / "rve2_hpo_pareto_front.csv"
pareto_df.to_csv(pareto_path, index=False)

print(f"✅ Pareto Front saved")
print(f"   → {len(pareto_df)} non-dominated models")

✅ All trials saved
   → 60 trials total
✅ Pareto Front saved
   → 15 non-dominated models


In [12]:
import optuna.visualization as vis

# Optuna Pareto plot
fig_pareto = vis.plot_pareto_front(study, target_names=["Val Loss", "Params"])
fig_pareto.update_layout(height=500, title="DualEncoderFNO - Multi_Objective HPO (NSGA-II) - Pareto Front")
fig_pareto.show()

html_path = cfg.hpo_dir / "rve2_hpo_pareto_front.html"
fig_pareto.write_html(str(html_path))
print("✅ Pareto Front saved")

✅ Pareto Front saved


**Observations:**
- As parameters increase, validation loss decreases.
- A compromise between precision and number of parameters can be found at the knee of the Pareto curve with losses around 0.20 and number of parameters between 2 and 4 millons.

In [13]:
fig_history = vis.plot_optimization_history(
    study, 
    target=lambda t: t.values[0],
    target_name="Val Loss"
)
fig_history.update_layout(title="DualEncoderFNO - Multi_Objective HPO - Optimization history", height=500)
fig_history.show()

html_path = cfg.hpo_dir / "rve2_hpo_opt_history.html"
fig_history.write_html(str(html_path))
print("✅ Optimization History saved")

✅ Optimization History saved


In [14]:
# Parallel coordinates plot of the Pareto Front
fig_parallel = vis.plot_parallel_coordinate(
    study,
    target=lambda t: t.values[0],
    target_name="Val Loss"
)

fig_parallel.update_traces(
    line=dict(
        colorscale="YlOrRd_r",
        reversescale=True,
        colorbar=dict(
            title="Val Loss",
            thickness=25,
            len=0.9
        )
    )
)

fig_parallel.update_layout(title="DualEncoderFNO - Multi_Objective HPO - Parallel Coordinates", height=400)
fig_parallel.show()

parcoord_html_path = cfg.hpo_dir / "rve2_hpo_parallel_coords.html"
fig_parallel.write_html(str(parcoord_html_path))
print("✅ Parallel Coordinates saved")

✅ Parallel Coordinates saved


In [15]:
fig_importance = vis.plot_param_importances(
    study, 
    target=lambda t: t.values[0],
    target_name="Val Loss"
)
fig_importance.update_layout(height=400, title="DualEncoderFNO - Multi_Objective HPO - Param importance in Loss")
fig_importance.show()
fig_importance.write_html(str(cfg.hpo_dir / "rve2_hpo_param_importance.html"))
print("✅ Parameter Importance saved")

✅ Parameter Importance saved


In [16]:
fig_importance_params = vis.plot_param_importances(
    study, 
    target=lambda t: t.values[1],
    target_name="N Params"
)
fig_importance_params.update_layout(height=400, title="DualEncoderFNO - Multi_Objective HPO - Param importance in NPARAMS")
fig_importance_params.show()
fig_importance_params.write_html(str(cfg.hpo_dir / "rve2_hpo_param_importance_NPARAMS.html"))
print("✅ Parameter Importance saved")

✅ Parameter Importance saved


**Key Observations:**

- `n_modes` is the **most influential hyperparameter** for achieving lower validation loss, while having **moderate influence** on model size compared to other parameters.
- `hidden_channels` contributes significantly to both validation loss reduction and model size increase, but it is the **primary driver** of parameter count.
- `n_layers`, `film_mlp_layers`, and `film_mlp_neurons` show **very low importance** in both validation loss and model size within the explored range.

**Conclusion and Recommendation:**

- **Prioritize exploring higher values of `n_modes`** over increasing `hidden_channels`.  
  This offers the best trade-off: significant improvements in accuracy with a more moderate increase in model size.

- Keep `hidden_channels` in a **moderate range** to avoid excessive parameter growth.
- **Fix or lightly explore** `n_layers` as well as FiLM MLP parameters, since they contribute very little to either objective in the current setup.